# CPU Figures: 4.3 Coarsening, 4.8 Spinodal, 4.6 Convergence

This notebook embeds the CPU scripts.
Run this notebook from the repository root. Generated figures are written to `output/`.For long simulations, start with the CPU coarsening example before running the full spinodal or convergence studies.


## Figure 4.3 Coarsening (CPU)


In [ ]:
"""Coarsening test for Figure 4.3 (Cahn-Hilliard-Darcy)"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft2, ifft2, fftfreq
import os

# Ensure output directory exists
os.makedirs("output", exist_ok=True)

class CahnHilliardDarcySolver:
    """
    Corrected Solver for the Cahn-Hilliard-Darcy system (Yang 2021).
    Corrections: H = f(phi)/sqrt(...); mu = lambda*(-Lap(phi) + H*U).
    """
    def __init__(self, Lx=2*np.pi, Ly=2*np.pi, Nx=128, Ny=128, dt=0.001,
                 alpha=100.0, M=1.0, lambda_param=0.01,
                 epsilon=0.05, S=2.0, tau=1.0, B=10.0):

        # Parameters
        self.Lx, self.Ly = Lx, Ly
        self.Nx, self.Ny = Nx, Ny
        self.dt = dt
        self.alpha = alpha
        self.M = M
        self.lam = lambda_param
        self.eps = epsilon
        self.S = S
        self.tau = tau
        self.B = B

        # Spatial grid and Fourier wavenumbers
        self.dx = Lx / Nx
        self.dy = Ly / Ny
        self.x = np.linspace(0, Lx, Nx, endpoint=False)
        self.y = np.linspace(0, Ly, Ny, endpoint=False)
        self.X, self.Y = np.meshgrid(self.x, self.y, indexing='ij')

        # Spectral Grid
        self.kx = 2 * np.pi * fftfreq(Nx, d=self.dx)
        self.ky = 2 * np.pi * fftfreq(Ny, d=self.dy)
        self.KX, self.KY = np.meshgrid(self.kx, self.ky, indexing='ij')
        self.K2 = self.KX**2 + self.KY**2
        self.K2[0,0] = 1e-10 # Avoid division by zero

        # Fields
        self.phi = np.zeros((Nx, Ny))
        self.phi_old = np.zeros((Nx, Ny))
        self.u = np.zeros((Nx, Ny))
        self.v = np.zeros((Nx, Ny))
        self.p = np.zeros((Nx, Ny))
        self.mu = np.zeros((Nx, Ny))

        # SAV variable U
        self.U = 0.0
        self.t = 0.0

    def init_two_circles(self):
        """Initial condition: two tanh-profile circles (Fig. 4.3)"""
        x1, y1 = np.pi - 0.8, np.pi
        x2, y2 = np.pi + 1.7, np.pi
        r1, r2 = 1.4, 0.5

        dist1 = np.sqrt((self.X - x1)**2 + (self.Y - y1)**2)
        dist2 = np.sqrt((self.X - x2)**2 + (self.Y - y2)**2)

        # Tanh profile setup
        self.phi = 1.0 + np.tanh((r1 - dist1)/(1.5*self.eps)) + \
                         np.tanh((r2 - dist2)/(1.5*self.eps))
        self.phi_old = self.phi.copy()
        self._init_sav()

    def _init_sav(self):
        F = (0.25 / self.eps**2) * (self.phi**2 - 1)**2
        E_bulk = np.sum(F) * self.dx * self.dy
        self.U = np.sqrt(E_bulk + self.B)

    def step(self):
        """Perform one SAV time step (Cahn-Hilliard + Darcy)"""

        # --- Cahn-Hilliard step ---

        # Extrapolate phi for nonlinearity (2nd order BDF-like)
        phi_star = 2.0 * self.phi - self.phi_old

        # A. Compute H (Nonlinear part ONLY)
        # f(phi) = (phi^3 - phi)/eps^2
        f_phi = (1.0 / self.eps**2) * (phi_star**3 - phi_star)

        # Calculate energy integral for H denominator
        F_term = (0.25 / self.eps**2) * (phi_star**2 - 1)**2
        E_integral = np.sum(F_term) * self.dx * self.dy

        # H = f(phi) / sqrt(...) (no Laplacian here)
        H = self.lam * f_phi / np.sqrt(E_integral + self.B)

        # B. Solve for phi^(n+1) in Fourier space (implicit linear solve)

        phi_hat = fft2(self.phi)
        phi_old_hat = fft2(self.phi_old)

        # Advection: u · grad(phi)
        grad_phi_x = np.real(ifft2(1j * self.KX * phi_hat))
        grad_phi_y = np.real(ifft2(1j * self.KY * phi_hat))
        advection = self.u * grad_phi_x + self.v * grad_phi_y
        adv_hat = fft2(advection)

        # Stabilization coeff S
        stab_coeff = self.S / self.eps**2

        # LHS Operator (Implicit)
        # Note: Lap(Lap(phi)) -> K^4, Lap(phi) -> -K^2
        lhs_op = (1.5/self.dt) + self.M * self.lam * self.K2**2 + self.M * stab_coeff * self.K2

        # RHS time term (BDF2)
        rhs_time = (2.0 * phi_hat - 0.5 * phi_old_hat) / self.dt

        # Forcing from SAV: -M*K2*(lam*H*U - stab*phi_star)
        forcing_spatial = H * self.U - stab_coeff * phi_star
        forcing_hat = fft2(forcing_spatial)
        rhs_spatial = -self.M * self.K2 * forcing_hat

        rhs_total = rhs_time - adv_hat + rhs_spatial

        phi_new_hat = rhs_total / lhs_op
        phi_new = np.real(ifft2(phi_new_hat))

        # Update U (SAV): U_t = 0.5 * ∫ H * phi_t
        diff_phi = phi_new - self.phi
        integral_update = 0.5 * np.sum(H * diff_phi) * self.dx * self.dy
        self.U = self.U + integral_update

        # --- Darcy step ---

        # Chemical potential mu = lambda * (-Lap(phi) + H*U)
        lap_phi_new = np.real(ifft2(-self.K2 * fft2(phi_new)))
        # Recalculate H with new phi for best accuracy
        f_phi_new = (1.0 / self.eps**2) * (phi_new**3 - phi_new)
        F_new = (0.25 / self.eps**2) * (phi_new**2 - 1)**2
        E_new = np.sum(F_new) * self.dx * self.dy
        H_new = self.lam * f_phi_new / np.sqrt(E_new + self.B)

        self.mu = self.lam * (-lap_phi_new + H_new * self.U)

        # Momentum: (tau/dt + alpha) u + grad p = RHS, projected to divergence-free
        mu_hat = fft2(self.mu)
        grad_mu_x = np.real(ifft2(1j * self.KX * mu_hat))
        grad_mu_y = np.real(ifft2(1j * self.KY * mu_hat))

        force_x = -phi_new * grad_mu_x
        force_y = -phi_new * grad_mu_y

        coeff_u = (self.tau / self.dt) + self.alpha
        rhs_u = (self.tau / self.dt) * self.u + force_x
        rhs_v = (self.tau / self.dt) * self.v + force_y

        # Projection
        div_rhs = 1j * self.KX * fft2(rhs_u) + 1j * self.KY * fft2(rhs_v)
        p_hat = div_rhs / (-self.K2 * coeff_u / coeff_u) * (1.0/coeff_u)
        # Simplified: Lap(p) = div(RHS/coeff) * coeff
        # p_hat = div(RHS) / (-K^2)
        p_hat = div_rhs / (-self.K2)
        p_hat[0,0] = 0.0

        grad_p_x = np.real(ifft2(1j * self.KX * p_hat))
        grad_p_y = np.real(ifft2(1j * self.KY * p_hat))

        self.u = (rhs_u - grad_p_x) / coeff_u
        self.v = (rhs_v - grad_p_y) / coeff_u

        # Update state
        self.phi_old = self.phi.copy()
        self.phi = phi_new
        self.t += self.dt

def run_case(label, solver, target_times, save_name):
    print(f"--- Running {label} ---")
    snapshots = []

    # Capture t=0
    if 0.0 in target_times:
        snapshots.append((0.0, solver.phi.copy()))

    max_time = max(target_times)
    # BDF2 needs a small kickstart or just run (steps will handle self.n=0 logic implicitly by variable init)
    # Note: The step function assumes BDF2 logic. For t=0->1, error is small.

    steps = int(max_time / solver.dt) + 20

    for i in range(steps):
        solver.step()

        # Check times
        for target in target_times:
            if abs(solver.t - target) < solver.dt * 0.6:
                print(f"  Saving t={target:.2f}")
                snapshots.append((target, solver.phi.copy()))

        if solver.t > max_time + solver.dt:
            break

    # Visualization
    if not snapshots: return

    cols = min(len(snapshots), 6)
    rows = (len(snapshots) - 1) // 6 + 1
    fig, axes = plt.subplots(rows, cols, figsize=(3*cols, 3*rows + 0.5))

    if rows == 1 and cols == 1: axes = [axes]
    elif rows > 1 or cols > 1: axes = axes.flatten()

    for i, (t_val, phi_val) in enumerate(snapshots):
        ax = axes[i]
        # 'jet' colormap to reproduce "ochra" (red/yellow) bubbles and blue background
        cf = ax.contourf(solver.X, solver.Y, phi_val,
                         levels=np.linspace(-1.1, 1.1, 50), cmap='jet')
        # Interface line
        ax.contour(solver.X, solver.Y, phi_val, levels=[0], colors='white', linewidths=1)
        ax.set_title(f"t={t_val:.2f}")
        ax.axis('off')
        ax.set_aspect('equal')

    for j in range(i+1, len(axes)): axes[j].axis('off')

    plt.suptitle(label)
    plt.tight_layout()
    plt.savefig(f"output/{save_name}.png", dpi=150)
    plt.close()
    print(f"Saved {save_name}.png")

if __name__ == "__main__":
    # --- Figure 4.3 ---
    # Parameters [cite: 546]
    # Note: Using slightly larger dt to reach t=5 faster, stability provided by SAV.
    solver43 = CahnHilliardDarcySolver(
        Lx=2*np.pi, Ly=2*np.pi, Nx=128, Ny=128, dt=0.005,
        alpha=100.0, M=1.0, lambda_param=0.01,
        epsilon=0.05, S=2.0, tau=1.0
    )
    solver43.init_two_circles()
    times43 = [0.0, 1.4, 1.9, 1.95, 2.45, 5.0]
    run_case("Figure 4.3: Coarsening", solver43, times43, "Figure_4_3_Fixed_Color")


--- Running Figure 4.3: Coarsening ---
  Saving t=1.40
  Saving t=1.90
  Saving t=1.95
  Saving t=2.45
  Saving t=5.00
Saved Figure_4_3_Fixed_Color.png


## Figure 4.8 Spinodal (CPU)


In [ ]:
"""Spinodal decomposition driver for Fig. 4.8 (CPU)."""

import numpy as np
import matplotlib.pyplot as plt
import os

# FFT backend: prefer pyFFTW (threaded), fallback to scipy.fft
USE_PYFFTW = False
FFT_THREADS = max(1, min(8, (os.cpu_count() or 1)))
try:
    import pyfftw
    from pyfftw.interfaces.numpy_fft import fft2 as fft2_base, ifft2 as ifft2_base, fftfreq

    pyfftw.interfaces.cache.enable()
    USE_PYFFTW = True
except ImportError:
    from scipy.fft import fft2 as fft2_base, ifft2 as ifft2_base, fftfreq

def fft2_wrap(a):
    return fft2_base(a, threads=FFT_THREADS) if USE_PYFFTW else fft2_base(a)

def ifft2_wrap(a):
    return ifft2_base(a, threads=FFT_THREADS) if USE_PYFFTW else ifft2_base(a)

# Ensure output directory exists
os.makedirs("output", exist_ok=True)

class CahnHilliardDarcySolver:
    """
    Corrected Solver for the Cahn-Hilliard-Darcy system (Yang 2021).

    Fixes applied:
    1. Applied lambda scaling to the nonlinear and stabilization terms in the CH step.
    2. Corrected LHS operator to include lambda in the stabilization term.
    """
    def __init__(self, Lx=2*np.pi, Ly=2*np.pi, Nx=512, Ny=512, dt=0.001,
                 alpha=100.0, M=1.0, lambda_param=0.01,
                 epsilon=0.025, S=10.0, tau=1.0, B=10.0):

        # Parameters
        self.Lx, self.Ly = Lx, Ly
        self.Nx, self.Ny = Nx, Ny
        self.dt = dt
        self.alpha = alpha
        self.M = M
        self.lam = lambda_param
        self.eps = epsilon
        self.S = S
        self.tau = tau
        self.B = B

        # Spatial grid
        self.dx = Lx / Nx
        self.dy = Ly / Ny
        self.x = np.linspace(0, Lx, Nx, endpoint=False)
        self.y = np.linspace(0, Ly, Ny, endpoint=False)
        self.X, self.Y = np.meshgrid(self.x, self.y, indexing='ij')

        # Spectral grid (Fourier wavenumbers)
        self.kx = 2 * np.pi * fftfreq(Nx, d=self.dx)
        self.ky = 2 * np.pi * fftfreq(Ny, d=self.dy)
        self.KX, self.KY = np.meshgrid(self.kx, self.ky, indexing='ij')
        self.K2 = self.KX**2 + self.KY**2
        self.K2[0,0] = 1e-10 # Avoid division by zero

        # Fields
        self.phi = np.zeros((Nx, Ny))
        self.phi_old = np.zeros((Nx, Ny))
        self.u = np.zeros((Nx, Ny))
        self.v = np.zeros((Nx, Ny))
        self.p = np.zeros((Nx, Ny))
        self.mu = np.zeros((Nx, Ny))

        # SAV variable U
        self.U = 0.0
        self.t = 0.0

    def init_two_circles(self):
        """Initial condition: two tanh-profile disks (Fig. 4.3)."""
        x1, y1 = np.pi - 0.8, np.pi
        x2, y2 = np.pi + 1.7, np.pi
        r1, r2 = 1.4, 0.5

        dist1 = np.sqrt((self.X - x1)**2 + (self.Y - y1)**2)
        dist2 = np.sqrt((self.X - x2)**2 + (self.Y - y2)**2)

        # Tanh profile setup
        self.phi = 1.0 + np.tanh((r1 - dist1)/(1.5*self.eps)) + \
                         np.tanh((r2 - dist2)/(1.5*self.eps))
        self.phi_old = self.phi.copy()
        self._init_sav()

    def init_spinodal(self, phi_avg, noise_amp=0.001):
        """Spinodal start with small random noise (paper: 0.001*(2*(rand-0.5)))."""
        # Initial condition: phi_avg + noise_amp * random perturbation
        noise = noise_amp * (2 * np.random.rand(self.Nx, self.Ny) - 1)
        self.phi = phi_avg + noise
        self.phi_old = self.phi.copy()
        self._init_sav()

    def _init_sav(self):
        # Energy functional F(phi)
        F = (0.25 / self.eps**2) * (self.phi**2 - 1)**2
        E_bulk = np.sum(F) * self.dx * self.dy
        self.U = np.sqrt(E_bulk + self.B)

    def save_state(self, path):
        """Checkpoint current state to an .npz file."""
        np.savez(
            path,
            phi=self.phi,
            phi_old=self.phi_old,
            u=self.u,
            v=self.v,
            p=self.p,
            mu=self.mu,
            U=self.U,
            t=self.t,
        )

    def load_state(self, path):
        """Load state from an .npz checkpoint."""
        data = np.load(path)
        self.phi = data["phi"]
        self.phi_old = data["phi_old"]
        self.u = data["u"]
        self.v = data["v"]
        self.p = data["p"]
        self.mu = data["mu"]
        self.U = float(data["U"])
        self.t = float(data["t"])

    def step(self):
        """Perform one SAV time step (Cahn-Hilliard + Darcy)."""
        # --- Cahn-Hilliard step ---

        # Extrapolate phi for nonlinearity (2nd order BDF-like)
        phi_star = 2.0 * self.phi - self.phi_old

        # A. Compute H (nonlinear part only)
        # f(phi) = (phi^3 - phi)/eps^2
        f_phi = (1.0 / self.eps**2) * (phi_star**3 - phi_star)

        # Calculate energy integral for H denominator
        F_term = (0.25 / self.eps**2) * (phi_star**2 - 1)**2
        E_integral = np.sum(F_term) * self.dx * self.dy

        # H = f(phi) / sqrt(...)
        H = f_phi / np.sqrt(E_integral + self.B)

        # B. Solve for phi^(n+1) in Fourier space

        phi_hat = fft2_wrap(self.phi)
        phi_old_hat = fft2_wrap(self.phi_old)

        # Advection: u � grad(phi)
        grad_phi_x = np.real(ifft2_wrap(1j * self.KX * phi_hat))
        grad_phi_y = np.real(ifft2_wrap(1j * self.KY * phi_hat))
        advection = self.u * grad_phi_x + self.v * grad_phi_y
        adv_hat = fft2_wrap(advection)

        # Stabilization coeff S
        stab_coeff = self.S / self.eps**2

        # LHS Operator (Implicit)
        # Equation 3.18: mu = lambda * (-Lap(phi) + H*U + S/eps^2(phi - phi*))
        # Substitute into phi_t = M * Lap(mu)
        # The Stabilizer term S/eps^2 is also multiplied by lambda!
        # Term: M * Lap ( lambda * S/eps^2 * phi ) -> M * lambda * S/eps^2 * (-K2) * phi
        lhs_op = (1.5/self.dt) + \
                 self.M * self.lam * self.K2**2 + \
                 self.M * self.lam * stab_coeff * self.K2

        # RHS:
        # 1. Time terms (BDF2: (2phi - 0.5phi_old)/dt)
        rhs_time = (2.0 * phi_hat - 0.5 * phi_old_hat) / self.dt

        # 2. Forcing from SAV
        # Explicit part of mu: lambda * (H*U - S/eps^2 * phi_star)
        # Laplacian applied to it: M * Lap( ... ) -> -M * K2 * ...
        forcing_spatial = self.lam * (H * self.U - stab_coeff * phi_star) # FIXED: Added self.lam
        forcing_hat = fft2_wrap(forcing_spatial)
        rhs_spatial = -self.M * self.K2 * forcing_hat

        rhs_total = rhs_time - adv_hat + rhs_spatial

        phi_new_hat = rhs_total / lhs_op
        phi_new = np.real(ifft2_wrap(phi_new_hat))

        # C. Update U (SAV)
        # U_t = 0.5 * int(H * phi_t)
        diff_phi = phi_new - self.phi
        integral_update = 0.5 * np.sum(H * diff_phi) * self.dx * self.dy
        self.U = self.U + integral_update

        # --- 2. Darcy Step ---

        # Update Chemical Potential mu for Darcy force
        # mu = lambda * (-Lap(phi) + H*U) (Using consistent SAV potential)
        lap_phi_new = np.real(ifft2_wrap(-self.K2 * fft2_wrap(phi_new)))

        # Recalculate H with new phi for best accuracy
        f_phi_new = (1.0 / self.eps**2) * (phi_new**3 - phi_new)
        F_new = (0.25 / self.eps**2) * (phi_new**2 - 1)**2
        E_new = np.sum(F_new) * self.dx * self.dy
        H_new = f_phi_new / np.sqrt(E_new + self.B)

        # FIXED: mu definition requires lambda scaling on ALL terms
        self.mu = self.lam * (-lap_phi_new + H_new * self.U)

        # Solve Momentum: (tau/dt + alpha) u + grad p = RHS
        mu_hat = fft2_wrap(self.mu)
        grad_mu_x = np.real(ifft2_wrap(1j * self.KX * mu_hat))
        grad_mu_y = np.real(ifft2_wrap(1j * self.KY * mu_hat))

        force_x = -phi_new * grad_mu_x
        force_y = -phi_new * grad_mu_y

        coeff_u = (self.tau / self.dt) + self.alpha
        rhs_u = (self.tau / self.dt) * self.u + force_x
        rhs_v = (self.tau / self.dt) * self.v + force_y

        # Projection
        div_rhs = 1j * self.KX * fft2_wrap(rhs_u) + 1j * self.KY * fft2_wrap(rhs_v)
        p_hat = div_rhs / (-self.K2)
        p_hat[0,0] = 0.0

        grad_p_x = np.real(ifft2_wrap(1j * self.KX * p_hat))
        grad_p_y = np.real(ifft2_wrap(1j * self.KY * p_hat))

        self.u = (rhs_u - grad_p_x) / coeff_u
        self.v = (rhs_v - grad_p_y) / coeff_u

        # Update state
        self.phi_old = self.phi.copy()
        self.phi = phi_new
        self.t += self.dt

def run_case(label, solver, target_times, save_name):
    print(f"--- Running {label} ---")
    snapshots = []
    captured = set()

    # Capture t=0 if requested
    if 0.0 in target_times:
        snapshots.append((0.0, solver.phi.copy()))
        captured.add(0.0)

    max_time = max(target_times)
    steps = int(max_time / solver.dt) + 50

    for _ in range(steps):
        solver.step()

        # Check times
        for target in target_times:
            if target in captured:
                continue
            if abs(solver.t - target) < solver.dt * 0.6:
                print(f"  Saving t={target:.2f}")
                snapshots.append((target, solver.phi.copy()))
                captured.add(target)

        if solver.t > max_time + solver.dt:
            break

    if not snapshots:
        return

    # Sort snapshots by time before plotting
    snapshots.sort(key=lambda x: x[0])

    cols = min(len(snapshots), 5)  # Figure 4.8 has 5 cols
    rows = (len(snapshots) - 1) // 5 + 1
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows + 0.5))

    if rows == 1 and cols == 1:
        axes = [axes]
    elif rows > 1 or cols > 1:
        axes = axes.flatten()

    if len(axes) > len(snapshots):
        for j in range(len(snapshots), len(axes)):
            axes[j].axis('off')

    for i, (t_val, phi_val) in enumerate(snapshots):
        ax = axes[i]
        cf = ax.contourf(
            solver.X,
            solver.Y,
            phi_val,
            levels=np.linspace(-1.1, 1.1, 50),
            cmap="jet",
        )
        ax.contour(solver.X, solver.Y, phi_val, levels=[0], colors="white", linewidths=1)
        ax.set_title(f"t={t_val:.1f}")
        ax.axis("off")
        ax.set_aspect("equal")

    plt.suptitle(label)
    plt.tight_layout()
    plt.savefig(f"output/{save_name}.png", dpi=150)
    plt.close()
    print(f"Saved {save_name}.png")

if __name__ == "__main__":
    print("Initializing Figure 4.8a...")
    solver48a = CahnHilliardDarcySolver(
        Lx=2*np.pi, Ly=2*np.pi, Nx=512, Ny=512, dt=0.001,
        alpha=100.0, M=1.0, lambda_param=0.01,
        epsilon=0.025, S=10.0, tau=1.0
    )
    solver48a.init_spinodal(phi_avg=0.0, noise_amp=0.001)
    times48a = [0.5, 5.0, 10.0, 15.0, 40.0]
    run_case("Figure 4.8a: Spinodal (phi=0)", solver48a, times48a, "Figure_4_8a_Fixed")

    print("Initializing Figure 4.8b...")
    solver48b = CahnHilliardDarcySolver(
        Lx=2*np.pi, Ly=2*np.pi, Nx=512, Ny=512, dt=0.001,
        alpha=100.0, M=1.0, lambda_param=0.01,
        epsilon=0.025, S=10.0, tau=1.0
    )
    solver48b.init_spinodal(phi_avg=0.3, noise_amp=0.001)
    times48b = [5.0, 10.0, 15.0, 40.0, 50.0]
    run_case("Figure 4.8b: Spinodal (phi=0.3)", solver48b, times48b, "Figure_4_8b_Fixed")


Initializing Figure 4.8a...
--- Running Figure 4.8a: Spinodal (phi=0) ---
  Saving t=0.50
  Saving t=5.00
  Saving t=10.00
  Saving t=15.00
  Saving t=40.00
Saved Figure_4_8a_Fixed.png
Initializing Figure 4.8b...
--- Running Figure 4.8b: Spinodal (phi=0.3) ---
  Saving t=5.00
  Saving t=10.00
  Saving t=15.00
  Saving t=40.00
  Saving t=50.00
Saved Figure_4_8b_Fixed.png


## Figure 4.6 Convergence (CPU)


In [ ]:
"""
Minimal driver to reproduce Fig. 4.6 (DSAV vs EX-SAV vs AV) for the
Cahn-Hilliard-Darcy system (Yang 2021). Uses initial condition (4.1),
parameters (4.2), and computes L2 errors against a small-step DSAV
reference. EX-SAV/AV remove SAV normalization (no Q); AV also sets S=0.
"""

import os
import numpy as np
import matplotlib.pyplot as plt

# FFT backend: prefer pyFFTW (threaded) else scipy.fft
USE_PYFFTW = False
FFT_THREADS = max(1, min(8, (os.cpu_count() or 1)))
try:
    import pyfftw
    from pyfftw.interfaces.numpy_fft import fft2 as fft2_base, ifft2 as ifft2_base, fftfreq

    pyfftw.interfaces.cache.enable()
    USE_PYFFTW = True
except ImportError:
    from scipy.fft import fft2 as fft2_base, ifft2 as ifft2_base, fftfreq


def fft2_wrap(a):
    return fft2_base(a, threads=FFT_THREADS) if USE_PYFFTW else fft2_base(a)


def ifft2_wrap(a):
    return ifft2_base(a, threads=FFT_THREADS) if USE_PYFFTW else ifft2_base(a)

from pathlib import Path

# Directories
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)


class CahnHilliardDarcySolver:
    """
    Solver with SAV toggle (use_sav=True for DSAV, False for EX-SAV/AV).
    Only the pieces needed for Figure 4.6 are kept.
    """

    def __init__(
        self,
        Lx=2 * np.pi,
        Ly=2 * np.pi,
        Nx=128,
        Ny=128,
        dt=0.001,
        alpha=100.0,
        M=1.0,
        lambda_param=0.01,
        epsilon=0.05,
        S=2.0,
        tau=1.0,
        B=10.0,
        use_sav=True,
    ):
        self.Lx, self.Ly = Lx, Ly
        self.Nx, self.Ny = Nx, Ny
        self.dt = dt
        self.alpha = alpha
        self.M = M
        self.lam = lambda_param
        self.eps = epsilon
        self.S = S
        self.tau = tau
        self.B = B
        self.use_sav = use_sav

        # Spatial grid
        self.dx = Lx / Nx
        self.dy = Ly / Ny
        self.x = np.linspace(0, Lx, Nx, endpoint=False)
        self.y = np.linspace(0, Ly, Ny, endpoint=False)
        self.X, self.Y = np.meshgrid(self.x, self.y, indexing="ij")

        # Spectral grid (Fourier wavenumbers)
        self.kx = 2 * np.pi * fftfreq(Nx, d=self.dx)
        self.ky = 2 * np.pi * fftfreq(Ny, d=self.dy)
        self.KX, self.KY = np.meshgrid(self.kx, self.ky, indexing="ij")
        self.K2 = self.KX**2 + self.KY**2
        self.K2[0, 0] = 1e-10  # avoid divide by zero

        # Fields
        self.phi = np.zeros((Nx, Ny))
        self.phi_old = np.zeros((Nx, Ny))
        self.u = np.zeros((Nx, Ny))
        self.v = np.zeros((Nx, Ny))
        self.mu = np.zeros((Nx, Ny))
        self.U = 0.0  # SAV variable
        self.t = 0.0

    def init_two_circles(self):
        """Initial condition (4.1): two tanh-profile disks."""
        x1, y1 = np.pi - 0.8, np.pi
        x2, y2 = np.pi + 1.7, np.pi
        r1, r2 = 1.4, 0.5
        dist1 = np.sqrt((self.X - x1) ** 2 + (self.Y - y1) ** 2)
        dist2 = np.sqrt((self.X - x2) ** 2 + (self.Y - y2) ** 2)
        self.phi = (
            1.0
            + np.tanh((r1 - dist1) / (1.5 * self.eps))
            + np.tanh((r2 - dist2) / (1.5 * self.eps))
        )
        self.phi_old = self.phi.copy()
        self._init_sav()

    def _init_sav(self):
        if not self.use_sav:
            # EX-SAV / AV: fix Q^n = 1 (no SAV normalization)
            self.U = 1.0
            return
        F = (0.25 / self.eps**2) * (self.phi**2 - 1) ** 2
        E_bulk = np.sum(F) * self.dx * self.dy
        self.U = np.sqrt(E_bulk + self.B)

    def step(self):
        # Extrapolate phi (2nd order BDF-like)
        phi_star = 2.0 * self.phi - self.phi_old

        # Nonlinear part (f = (phi^3-phi)/eps^2)
        f_phi = (1.0 / self.eps**2) * (phi_star**3 - phi_star)
        if self.use_sav:
            F_term = (0.25 / self.eps**2) * (phi_star**2 - 1) ** 2
            E_integral = np.sum(F_term) * self.dx * self.dy
            H = f_phi / np.sqrt(E_integral + self.B)
        else:
            # No-Q: standard nonlinear term without SAV scaling; AV also sets S=0.
            # This matches μ = λ(-Δφ + (φ^3-φ)/ε^2); AV also sets S=0 via caller.
            H = f_phi

        phi_hat = fft2_wrap(self.phi)
        phi_old_hat = fft2_wrap(self.phi_old)

        # Advection: u � grad(phi)
        grad_phi_x = np.real(ifft2_wrap(1j * self.KX * phi_hat))
        grad_phi_y = np.real(ifft2_wrap(1j * self.KY * phi_hat))
        advection = self.u * grad_phi_x + self.v * grad_phi_y
        adv_hat = fft2_wrap(advection)

        stab_coeff = self.S / self.eps**2

        # LHS operator (implicit)
        lhs_op = (1.5 / self.dt) + self.M * self.lam * self.K2**2 + self.M * self.lam * stab_coeff * self.K2

        # RHS components: time (BDF2) + forcing
        rhs_time = (2.0 * phi_hat - 0.5 * phi_old_hat) / self.dt
        forcing_spatial = self.lam * (H * self.U - stab_coeff * phi_star)
        forcing_hat = fft2_wrap(forcing_spatial)
        rhs_spatial = -self.M * self.K2 * forcing_hat
        rhs_total = rhs_time - adv_hat + rhs_spatial

        phi_new_hat = rhs_total / lhs_op
        phi_new = np.real(ifft2_wrap(phi_new_hat))

        # Update SAV
        if self.use_sav:
            diff_phi = phi_new - self.phi
            integral_update = 0.5 * np.sum(H * diff_phi) * self.dx * self.dy
            self.U += integral_update

        # Darcy step (velocity used for advection consistency)
        lap_phi_new = np.real(ifft2_wrap(-self.K2 * fft2_wrap(phi_new)))
        f_phi_new = (1.0 / self.eps**2) * (phi_new**3 - phi_new)
        if self.use_sav:
            F_new = (0.25 / self.eps**2) * (phi_new**2 - 1) ** 2
            E_new = np.sum(F_new) * self.dx * self.dy
            H_new = f_phi_new / np.sqrt(E_new + self.B)
            mu_explicit = H_new * self.U
        else:
            mu_explicit = f_phi_new  # no SAV scaling, no Q
        self.mu = self.lam * (-lap_phi_new + mu_explicit)

        mu_hat = fft2_wrap(self.mu)
        grad_mu_x = np.real(ifft2_wrap(1j * self.KX * mu_hat))
        grad_mu_y = np.real(ifft2_wrap(1j * self.KY * mu_hat))
        force_x = -phi_new * grad_mu_x
        force_y = -phi_new * grad_mu_y

        coeff_u = (self.tau / self.dt) + self.alpha
        rhs_u = (self.tau / self.dt) * self.u + force_x
        rhs_v = (self.tau / self.dt) * self.v + force_y

        div_rhs = 1j * self.KX * fft2_wrap(rhs_u) + 1j * self.KY * fft2_wrap(rhs_v)
        p_hat = div_rhs / (-self.K2)
        p_hat[0, 0] = 0.0
        grad_p_x = np.real(ifft2_wrap(1j * self.KX * p_hat))
        grad_p_y = np.real(ifft2_wrap(1j * self.KY * p_hat))

        self.u = (rhs_u - grad_p_x) / coeff_u
        self.v = (rhs_v - grad_p_y) / coeff_u

        self.phi_old = self.phi.copy()
        self.phi = phi_new
        self.t += self.dt


def l2_error(phi_num, phi_ref, dx, dy):
    diff = phi_num - phi_ref
    return np.sqrt(np.sum(diff**2) * dx * dy)


def compute_solution(dt, final_time, base_params, use_sav=True, S_override=None, label=None):
    params = base_params.copy()
    params.update(dict(dt=dt, use_sav=use_sav))
    if S_override is not None:
        params["S"] = S_override

    solver = CahnHilliardDarcySolver(**params)
    solver.init_two_circles()
    steps = int(np.ceil(final_time / dt))
    for _ in range(steps):
        solver.step()
    if label:
        print(f"  [{label}] t={solver.t:.5e}, steps={steps}")
    return solver.phi.copy(), solver.dx, solver.dy


def run_time_refinement_tests():
    base_params = dict(
        Lx=2 * np.pi,
        Ly=2 * np.pi,
        Nx=128,
        Ny=128,
        alpha=100.0,
        M=1.0,
        lambda_param=0.01,
        epsilon=0.05,
        S=2.0,
        tau=1.0,
        B=10.0,
    )

    # Use a longer horizon to separate DSAV vs EX-SAV; reference step remains small.
    final_time = 0.01
    dt_reference = 1e-9
    print(f"Computing DSAV reference (dt={dt_reference})...")
    phi_ref, dx_ref, dy_ref = compute_solution(
        dt_reference, final_time, base_params, use_sav=True, S_override=2.0, label="DSAV-ref"
    )

    dt_values = np.array([1e-2, 7e-3, 5e-3, 2e-3, 1e-3, 7e-4, 5e-4, 2e-4, 1e-4, 5e-5, 2e-5, 1e-5])

    errors_dsav = []
    errors_exsav = []
    errors_av = []

    for dt in dt_values:
        print(f"Running DSAV dt={dt}...")
        phi_dsav, dx, dy = compute_solution(dt, final_time, base_params, use_sav=True, S_override=2.0)
        errors_dsav.append(l2_error(phi_dsav, phi_ref, dx, dy))

        print(f"Running EX-SAV dt={dt}...")
        phi_exsav, dx, dy = compute_solution(dt, final_time, base_params, use_sav=False, S_override=2.0)
        errors_exsav.append(l2_error(phi_exsav, phi_ref, dx, dy))

        print(f"Running AV dt={dt}...")
        phi_av, dx, dy = compute_solution(dt, final_time, base_params, use_sav=False, S_override=0.0)
        errors_av.append(l2_error(phi_av, phi_ref, dx, dy))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # (a) DSAV vs EX-SAV
    axes[0].loglog(dt_values, errors_dsav, "rD-", label=r"DSAV:$\phi$")
    axes[0].loglog(dt_values, errors_exsav, "bd--", label=r"EX-SAV:$\phi$")
    axes[0].set_xlabel("Time Step")
    axes[0].set_ylabel(r"$L^2$ Error")
    axes[0].set_xlim(1e-5, 1e-2)
    axes[0].set_ylim(1e-7, 1e0)
    axes[0].legend()
    axes[0].grid(True, which="both", ls=":")
    axes[0].set_title("(a) DSAV and EX-SAV (no Q)")

    # (b) DSAV vs AV
    axes[1].loglog(dt_values, errors_dsav, "rD-", label=r"DSAV:$\phi$")
    axes[1].loglog(dt_values, errors_av, "bo-", label=r"AV:$\phi$")
    ref_y0 = errors_dsav[0]
    ref_line = ref_y0 * (dt_values / dt_values[0]) ** 2
    axes[1].loglog(dt_values, ref_line, "b--", label="Ref:slope 2")
    axes[1].set_xlabel("Time Step")
    axes[1].set_ylabel(r"$L^2$ Error")
    axes[1].set_xlim(1e-5, 1e-2)
    axes[1].set_ylim(1e-7, 1e0)
    axes[1].legend()
    axes[1].grid(True, which="both", ls=":")
    axes[1].set_title("(b) DSAV and AV (no S and Q)")

    plt.tight_layout()
    outfile = OUTPUT_DIR / "Figure_4_6_reproduction.png"
    plt.savefig(outfile, dpi=300)
    plt.close()
    print(f"Saved {outfile}")


if __name__ == "__main__":
    run_time_refinement_tests()
